In [1]:
import numpy as np
import pandas as pd
sr_scores = pd.Series([90, 85, 78, 92, 85], index=['Alice', 'Bob', 'Charlie', 'David', 'Eva'])
df_sales = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "product": ["Laptop", "Phone", "Tablet", "Laptop", "Phone", "Tablet", "Laptop", "Phone"],
    "price": [1000, 500, 300, 1200, 550, 320, 1100, 600],
    "quantity": [1, 2, 3, 1, 2, 1, 1, 2],
    "order_date": [
        "2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04",
        "2024-01-05", "2024-01-06", "2024-01-07", "2024-01-08"
    ]
})
df_employees = pd.DataFrame({
    "employee_id": [101, 102, 103, 104, 105, 106, 107, 108],
    "name": ["Alice", "Bob", "Charlie", "David", "Eva", "Frank", "Grace", "Helen"],
    "department": ["HR", "IT", "IT", "Finance", "HR", "IT", "Finance", "HR"],
    "age": [25, 32, 29, 41, 35, 28, 45, 30],
    "salary": [50000, 70000, 65000, 90000, 62000, 72000, 88000, 58000],
    "city": ["Hanoi", "HCMC", "Hanoi", "Danang", "HCMC", "Hanoi", "Danang", "HCMC"],
    "is_active": [True, True, False, True, True, False, True, True]
})

## Practice

11. Multiple aggregations with named output columns 
Group df_employees by department and use .agg() to produce a summary table with custom column names: 
•	employee_count 
•	avg_age 
•	avg_salary 
•	max_salary 
•	inactive_count 


In [2]:
df_employees.groupby('department').agg(
    employee_count = ('employee_id','count'),
    avg_age = ('age', 'mean'),
    avg_salary = ('salary', 'mean'),
    max_salary = ('salary', 'max'),
    inactive_count = ('is_active',lambda x : (~x).sum())
)

,employee_count,avg_age,avg_salary,max_salary,inactive_count
department,,,,,
Finance,2,43.000000,89000.000000,90000,0
HR,3,30.000000,56666.666667,62000,0
IT,3,29.666667,69000.000000,72000,2


12. Find the top-paying city within each department 
Group employees by both department and city, compute average salary, then determine which city has the highest average salary inside each department. 
 
Return one row per department


In [3]:
group = df_employees.groupby(['department', 'city'], as_index= False)['salary'].agg('mean')
group.loc[group.groupby('department')['salary'].idxmax()].reset_index(drop = True)


,department,city,salary
0,Finance,Danang,89000.0
1,HR,HCMC,60000.0
2,IT,HCMC,70000.0


30. Department efficiency challenge 
For each department, calculate: 
•	average salary 
•	average age 
•	active rate 
•	salary per age 
•	inactive count 
 
Then create a custom rule to classify departments: 
•	"High Efficiency" if active rate >= 0.75 and salary per age > 2000 
•	"Moderate Efficiency" if only one of those is true 
•	"Low Efficiency" otherwise 
Use grouped analysis and return a final department summary table with the classification. 


In [4]:
count_employees = len(df_employees['salary'])
result = df_employees.groupby('department').agg(
        average_salary = ('salary',lambda x : x.mean().round(1)),
        average_age = ('age', lambda x : x.mean().round(1)),
        active_rate = ('is_active', lambda x : x.mean()),
        inactive_count = ('is_active', lambda  x : (~x).sum())
    )
result['salary_per_age'] = (result['average_salary']/result['average_age']).round(1)
# result['high_efficiency'] = (result['active_rate'] >= 0.75) & (result['salary_per_age'] > 2000)
# result['morderate_efficiency'] = (result['active_rate'] >= 0.75) | (result['salary_per_age'] > 2000)
# result['low_efficiency'] = (result['active_rate'] < 0.75) & (result['salary_per_age'] <= 2000)
# result.drop(columns= ['high_efficiency','morderate_efficiency','low_efficiency'])
import numpy as np
result['classicfication'] = np.select(
    [(result['active_rate'] >= 0.75) & (result['salary_per_age'] > 2000),
    (result['active_rate'] >= 0.75) | (result['salary_per_age'] > 2000)],
    ['high_efficiency',
     'morderate_efficiency'],
    default = "low_efficiency"
)
result

,average_salary,average_age,active_rate,inactive_count,salary_per_age,classicfication
department,,,,,,
Finance,89000.0,43.0,1.000000,0,2069.8,high_efficiency
HR,56666.7,30.0,1.000000,0,1888.9,morderate_efficiency
IT,69000.0,29.7,0.333333,2,2323.2,morderate_efficiency


Filter df_employees to keep only departments that satisfy both: 
•	average salary greater than 60,000 
•	at least one inactive employee 
Return all original rows from those departments. 


In [5]:
df_employees.groupby('department').filter(lambda x :( x['salary'].mean() > 60000) & ((~x['salary']).any()))

,employee_id,name,department,age,salary,city,is_active
1,102,Bob,IT,32,70000,HCMC,True
2,103,Charlie,IT,29,65000,Hanoi,False
3,104,David,Finance,41,90000,Danang,True
5,106,Frank,IT,28,72000,Hanoi,False
6,107,Grace,Finance,45,88000,Danang,True


29. Multi-level sales ranking 
Create revenue = price * quantity, then group by product and calculate summary statistics. 
After that: 
•	rank products by total revenue 
•	rank products by total quantity sold 
•	rank products by average price 
Combine all ranks into one summary DataFrame. 


In [6]:
df_sales['revenue'] = df_sales['price']* df_sales['quantity']
df = df_sales.groupby('product').agg(
    total_revenue = ('revenue', 'sum'),
    total_quantity_sold = ('quantity','sum'),
    average_price = ('price','mean')
)
df_sales.drop(columns = 'revenue', axis = 1)
#df.sort_values(by = ['total_revenue','total_quantity_sold','average_price'])
df['total_revenue_rank'] = df['total_revenue'].rank(ascending= False, method = 'min').astype(int)
df['average_price_rank'] = df['average_price'].rank(ascending= False, method = 'min').astype(int)
df['total_quantity_sold_rank'] = df['total_quantity_sold'].rank(ascending = False,method = 'min').astype(int)
df


,total_revenue,total_quantity_sold,average_price,total_revenue_rank,average_price_rank,total_quantity_sold_rank
product,,,,,,
Laptop,3300,3,1100.0,1,1,3
Phone,3300,6,550.0,1,2,1
Tablet,1220,4,310.0,3,3,2


28. Custom group scoring with .apply() 
Define a custom “department strength score” as: 
strength = (average_salary * active_rate) / average_age 
Use .groupby().apply() to compute this score for each department, then sort from strongest to weakest


In [7]:
def department_strength_score(group):
    average_salary = round(group['salary'].mean(),1)
    active_rate = round(group['is_active'].sum()/len(group['is_active']),1)
    average_age = round(group['age'].mean(),1)
    strength = ((average_salary*active_rate)/average_age).round(1)
    return pd.Series({'department_strength_score' : strength})

df = df_employees.groupby('department').apply(department_strength_score)
df.sort_values(by = "department_strength_score", ascending= False)


C:\Users\vutun\AppData\Local\Temp\ipykernel_13308\2033653788.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df_employees.groupby('department').apply(department_strength_score)


,department_strength_score
department,
Finance,2069.8
HR,1888.9
IT,697.0


For each product group in df_sales, calculate total revenue and assign a label: 
•	"Excellent" if total revenue >= 2000 
•	"Good" if total revenue is from 1000 to 1999 
•	"Needs Improvement" otherwise 
Use .groupby().apply() to return a summary DataFrame


In [9]:
def classification(group):
    if (group['revenue'].sum() >= 2000):
        return  'Excellent'
    elif (group['revenue'].sum() >= 1000 and group['revenue'].sum() <= 1999):
        return "Good"
    else :
        return "Needs Improvement"

df_sales['revenue'] = df_sales['price'] * df_sales['quantity']
df = df_sales.groupby('product').apply(classification)
df

C:\Users\vutun\AppData\Local\Temp\ipykernel_13308\3596827420.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df_sales.groupby('product').apply(classification)


product
Laptop    Excellent
Phone     Excellent
Tablet         Good
dtype: object

26. Product concentration analysis 
For each product, compute the share of total quantity contributed by each order row. 
Then identify products where a single order contributes more than 50% of that product’s total quantity. 


In [14]:
df_sales['total of share'] = df_sales.groupby('product')['quantity'].transform(lambda x : (x/x.sum()).round(2))
df = df_sales.groupby('product').filter(lambda x : (x['total of share'] > 0.5).any())
df

,order_id,product,price,quantity,order_date,revenue,total of share
2,3,Tablet,300,3,2024-01-03,900,0.75
5,6,Tablet,320,1,2024-01-06,320,0.25
